# 01_segmentation.ipynb
## Purpose
Segment raw comments into sentence-like segments; produce seg_id and segment texts.

## Expected inputs
- `raw_inputs/data/dataset.csv (sanitized)`
- `raw_inputs/data/data_raw_no_duplikat.csv (sanitized)`

## Expected outputs
- `data/segments_index.csv (or equivalent segment table)`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

# Practical Segmentation for ABSA (Forum Text) + Audit

This notebook creates **segmentation practical** for comment forum yang long (Reddit/Kaskus), then saves:

1. **Segmentation result** → `dataset.csv`  
2. **Audit JSON** → `audit_segments.json`  
3. **Audit samples** (before/after) → `audit_samples_segments.csv`

## Prinsip segmentation
Pipeline segmentation (hirarkis):

1) **Sentence split**: split berdasarkan newline + `. ! ?`  
2) **Clause split** (jika still long): split berdasarkan discourse martors such as `tapi/namun/karena/jadi/...`  
3) **Fallback chunking**: jika still long juga, potong based jumlah word (sliding window + overlap)

Output segment cocok for:
- **rule-based parsing** (per segmentt)
- **sentiment per-aspect** with context yang lebih bersih
- **audit-ready** (ada `comment_id`, `seg_id`, `method`, long, dsb.)

> Dataset minimal memiliki column: `source`, `user`, `date`, `comments`  
> (if nama column beda, change di section konfigurasi).

In [ ]:
# =========================
# 0) Setup
# =========================
import re, json, random
from dataclasses import dataclass
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple, Optional

import pandas as pd

print("Ready.")

## 1) I/O configuration

Update the input path to match your file.  
Default outputs:
- `segments_absa.csv`
- `audit_segments.json`
- `audit_samples_segments.csv`

In [ ]:
# =========================
# 1) I/O
# =========================
INPUT_CSV = Path("data/data_raw_no_duplikat.csv")  # <-- ganti dengan file kamu
COL_SOURCE = "source"
COL_USER   = "user"
COL_DATE   = "date"
COL_TEXT   = "comments"

OUT_SEGMENTS = Path("data/dataset.csv")
OUT_AUDIT_JSON = Path("data/audit_segments.json")
OUT_AUDIT_SAMPLES = Path("data/audit_samples_segments.csv")

INPUT_CSV, OUT_SEGMENTS

## 2) Segmentation parameters


In [ ]:
# =========================
# 2) Segment config
# =========================
@dataclass
class SegConfig:
    # Normalization
    replace_url_token: bool = True
    replace_num_token: bool = True
    url_token: str = "<URL>"
    num_token: str = "<NUM>"

    # Clause split trigger
    clause_split_if_words: int = 30

    # Final chunking fallback
    max_words_per_segment: int = 60
    overlap_words: int = 12
    # Filter very short segments
    min_words_segment: int = 2

    # Safety
    max_segments_per_comment: int = 200  # mencegah komentar aneh menghasilkan segmen terlalu banyak

CFG = SegConfig()
CFG
CFG.max_words_per_segment = 30   # atau 30
CFG.overlap_words = 8           # 8–12 biasanya oke
CFG.clause_split_if_words = 25   # supaya clause split lebih agresif

## 3) Discourse markers for clause split


In [ ]:
# =========================
# 3) Clause markers
# =========================
CLAUSE_MARKERS = [
    # contrast
    "tapi", "tp", "namun", "cuman", "cuma", "sedangkan", "padahal", "walaupun", "meskipun", "meski",
    "di sisi lain", "on the other hand",
    # cause-effect
    "karena", "soalnya", "sebab", "jadi", "makanya", "sehingga", "akibatnya", "alhasil", "then",
    # addition / emphasis
    "selain itu", "terus", "bahkan", "belum lagi", "intinya", "btw", "by the way", "in short",
    # conditional
    "kalau", "jika", "bila", "unless", "when",
    # exemplify
    "misalnya", "contohnya", "for example", "e.g."
]

CLAUSE_MARKERS_SORTED = sorted(CLAUSE_MARKERS, key=len, reverse=True)
print("Markers:", len(CLAUSE_MARKERS_SORTED))

## 4) Normalisasi & sentence split



In [ ]:
# =========================
# 4) Normalization + sentence split
# =========================
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)?\b")
ODD_CHARS_RE = re.compile(r"[^0-9A-Za-zÀ-ÖØ-öø-ÿ\s\.\,\!\?\;\:\-\(\)\/\<\>\%]+")
WS_RE = re.compile(r"\s+")
SENT_BOUNDARY_RE = re.compile(r"(?<=[\.\!\?])\s+|\s+\n\s+")

def normalize_text(text: str, cfg: SegConfig) -> str:
    if text is None:
        return ""
    t = str(text).replace("\r", " ").replace("\n", " \n ")
    if cfg.replace_url_token:
        t = URL_RE.sub(f" {cfg.url_token} ", t)
    if cfg.replace_num_token:
        t = NUM_RE.sub(f" {cfg.num_token} ", t)
    t = ODD_CHARS_RE.sub(" ", t)
    t = WS_RE.sub(" ", t).strip()
    return t

def split_sentences(text: str) -> List[str]:
    t = text.strip()
    if not t:
        return []
    return [p.strip() for p in SENT_BOUNDARY_RE.split(t) if p and p.strip()]

def word_count(s: str) -> int:
    return len(s.split()) if s else 0

## 5) Clause split & chunking fallback

In [ ]:
# =========================
# 5) Clause split + chunking
# =========================
def clause_split(sentence: str, cfg: SegConfig) -> Tuple[List[str], List[str]]:
    s = sentence.strip()
    if not s:
        return [], []
    used = []
    s_low = s.lower()

    split_points = []
    for m in CLAUSE_MARKERS_SORTED:
        pat = re.compile(r"(?<!\w)" + re.escape(m) + r"(?!\w)")
        for hit in pat.finditer(s_low):
            split_points.append((hit.start(), m))
    if not split_points:
        return [s], []

    split_points.sort(key=lambda x: x[0])

    segments = []
    last = 0
    for idx, m in split_points:
        left = s[last:idx].strip()
        if word_count(left) >= max(6, cfg.min_words_segment + 2):
            segments.append(left)
            used.append(m)
            last = idx
    tail = s[last:].strip()
    if tail:
        segments.append(tail)

    merged = []
    for seg in segments:
        if not merged:
            merged.append(seg)
        else:
            if word_count(seg) < cfg.min_words_segment:
                merged[-1] = (merged[-1] + " " + seg).strip()
            else:
                merged.append(seg)
    return merged, used


def chunk_by_words(text: str, max_words: int, overlap: int) -> List[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text.strip()]
    step = max_words - overlap
    if step <= 0:
        step = max_words
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end == len(words):
            break
        start += step
    return chunks

## 6) Segmentation

In [ ]:
# =========================
# 6) Full segmentation
# =========================
def segment_comment(text: str, cfg: SegConfig) -> Tuple[List[Dict], Dict]:
    meta = {
        "raw_len_chars": len(str(text)) if text is not None else 0,
        "raw_len_words": word_count(str(text)) if text is not None else 0,
        "n_sentences": 0,
        "used_markers": {},
        "n_segments": 0,
    }

    norm = normalize_text(text, cfg)
    sents = split_sentences(norm)
    meta["n_sentences"] = len(sents)

    out = []
    marker_counter = Counter()

    for sent in sents:
        sent_wc = word_count(sent)
        if sent_wc == 0:
            continue

        if sent_wc >= cfg.clause_split_if_words:
            clauses, used = clause_split(sent, cfg)
            marker_counter.update(used)

            for c in clauses:
                c_wc = word_count(c)
                if c_wc < cfg.min_words_segment:
                    continue
                if c_wc > cfg.max_words_per_segment:
                    for ch in chunk_by_words(c, cfg.max_words_per_segment, cfg.overlap_words):
                        if word_count(ch) >= cfg.min_words_segment:
                            out.append({"seg_text": ch, "method": "chunk_after_clause", "word_len": word_count(ch)})
                else:
                    out.append({"seg_text": c, "method": "clause", "word_len": c_wc})
        else:
            if sent_wc > cfg.max_words_per_segment:
                for ch in chunk_by_words(sent, cfg.max_words_per_segment, cfg.overlap_words):
                    if word_count(ch) >= cfg.min_words_segment:
                        out.append({"seg_text": ch, "method": "chunk_after_sentence", "word_len": word_count(ch)})
            else:
                if sent_wc >= cfg.min_words_segment:
                    out.append({"seg_text": sent, "method": "sentence", "word_len": sent_wc})

    if len(out) > cfg.max_segments_per_comment:
        out = out[:cfg.max_segments_per_comment]

    meta["used_markers"] = dict(marker_counter)
    meta["n_segments"] = len(out)
    return out, meta

## 7) Run and Save

In [ ]:
# =========================
# 7) Run segmentation
# =========================
assert INPUT_CSV.exists(), f"Input file not found: {INPUT_CSV}"

df = pd.read_csv(INPUT_CSV).reset_index(drop=True)

for col in [COL_SOURCE, COL_USER, COL_DATE, COL_TEXT]:
    assert col in df.columns, f"Missing column '{col}'. Available: {list(df.columns)[:30]}"

df["comment_id"] = df.index.astype(int)

rows = []
method_counter = Counter()
marker_global = Counter()
audit_meta = []
seg_per_comment = []

INCLUDE_COMMENT_TEXT_IN_SEGMENTS = False  # set True jika ingin simpan comment_text juga

for _, r in df.iterrows():
    segs, meta = segment_comment(r[COL_TEXT], CFG)
    audit_meta.append(meta)
    seg_per_comment.append(meta["n_segments"])

    for j, s in enumerate(segs):
        method_counter[s["method"]] += 1
        rows.append({
            "comment_id": int(r["comment_id"]),
            "comment_ori": r[COL_TEXT],
            "seg_id": int(j),
            "source": r[COL_SOURCE],
            "user": r[COL_USER],
            "date": r[COL_DATE],
            **({"comment_text": str(r[COL_TEXT])} if INCLUDE_COMMENT_TEXT_IN_SEGMENTS else {}),
            "seg_text": s["seg_text"],
            "method": s["method"],
            "seg_word_len": int(s["word_len"]),
            "comment_word_len": int(meta["raw_len_words"]),
            "n_segments_in_comment": int(meta["n_segments"]),
        })

    for mk, c in meta["used_markers"].items():
        marker_global[mk] += c



segments_df = pd.DataFrame(rows)

segments_df.to_csv(OUT_SEGMENTS, index=False, encoding="utf-8")
print("[OK] Saved:", OUT_SEGMENTS, "| rows:", len(segments_df))




# audit
seg_counts = pd.Series(seg_per_comment)
total_segments = len(segments_df)

audit = {
    "input_file": str(INPUT_CSV),
    "rows_comments": int(len(df)),
    "rows_segments": int(total_segments),
    "config": CFG.__dict__,
    "segments_per_comment": {
        "mean": float(seg_counts.mean()) if len(df) else 0.0,
        "median": float(seg_counts.median()) if len(df) else 0.0,
        "p90": float(seg_counts.quantile(0.90)) if len(df) else 0.0,
        "p95": float(seg_counts.quantile(0.95)) if len(df) else 0.0,
        "max": int(seg_counts.max()) if len(df) else 0,
    },
    "methods": {m: int(c) for m, c in method_counter.most_common()},
    "top_clause_markers": marker_global.most_common(30),
}

OUT_AUDIT_JSON.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print("[OK] Saved:", OUT_AUDIT_JSON)

# audit samples
df_len = pd.DataFrame({
    "comment_id": df["comment_id"],
    "comment_word_len": [m["raw_len_words"] for m in audit_meta],
    "n_segments": [m["n_segments"] for m in audit_meta],
    "comment_text": df[COL_TEXT].fillna("").astype(str),
}).sort_values("comment_word_len", ascending=False)

samples = pd.concat([
    df_len.head(15),
    df_len.iloc[len(df_len)//2: len(df_len)//2 + 15],
    df_len.sample(n=min(20, len(df_len)), random_state=42),
], ignore_index=True).drop_duplicates(subset=["comment_id"])

seg_preview = segments_df.groupby("comment_id")["seg_text"].apply(lambda x: list(x.head(3))).reset_index()
samples = samples.merge(seg_preview, on="comment_id", how="left").rename(columns={"seg_text": "first3_segments"})
samples.to_csv(OUT_AUDIT_SAMPLES, index=False, encoding="utf-8")
print("[OK] Saved:", OUT_AUDIT_SAMPLES)

segments_df.head(10)

## 8) Quick checks

In [ ]:
print("Segments per comment (top 10):")
segments_df.groupby("comment_id").size().sort_values(ascending=False).head(10)

In [ ]:
print("Segment word length stats:")
segments_df["seg_word_len"].describe()

In [ ]:
print(len(segments_df))